Проверим метрики для прогнозов gpt

В yandexgpt грузили обучающую выборку (с нарушением баланса классов и обогащением малых классов всеми доступными примерами - см. gpt_dataset.ipynb) на 50 тысяч примеров (train_v4.json), которую использовали для дообучения ygpt в версии 5 и использовали тест на 900 примеров примерно по 100 на класс, но в тестах использовали тексты только назначений платежей, без обогоащения данныеми статей, проектов и доноров - проверить улучшит ли качество предсказания модель, дообученная на более широком объеме признаков, на исходных текстах. Результаты весьма скромные - на уровне 30%, модели явно не хватает в тестовых данных знаний о статьях, проектах и донорах.


In [1]:

import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option("display.max_colwidth", None)

y_true = pd.read_csv('/Users/roman_yakovlev/Documents/LemonPie/Сегментация_своя/gpt_classification/ver_7/y_test_balanced.csv', index_col=0)
y_pred_ygpt_ft = pd.read_csv('/Users/roman_yakovlev/Documents/LemonPie/Сегментация_своя/gpt_classification/ver_7/y_pred_ygpt_ft.csv',index_col=0)

In [2]:
# сверка индексов
print(y_true.index.equals(y_pred_ygpt_ft.index))

# проверка, что множества индексов совпадают (порядок не важен)
print(set(y_true.index) == set(y_pred_ygpt_ft.index))

# вывод различий
print("Есть в y_true, но нет в y_pred:", y_true.index.difference(y_pred_ygpt_ft.index))
print("Есть в y_pred, но нет в y_true:", y_pred_ygpt_ft.index.difference(y_true.index))

True
True
Есть в y_true, но нет в y_pred: Index([], dtype='int64')
Есть в y_pred, но нет в y_true: Index([], dtype='int64')


In [3]:
# считаем метрики для прогнозов YandexGPT дообученой на нашем датасете

accuracy = accuracy_score(y_true, y_pred_ygpt_ft)

precision = precision_score(y_true, y_pred_ygpt_ft, average='weighted')
recall = recall_score(y_true, y_pred_ygpt_ft, average='weighted')
f1 = f1_score(y_true, y_pred_ygpt_ft, average='weighted')

print("Accuracy:", accuracy)
print("Precision (weighted):", precision)
print("Recall (weighted):", recall)
print("F1-score (weighted):", f1)

print("\nОтчет по классам:")
print(classification_report(y_true, y_pred_ygpt_ft))

Accuracy: 0.2911111111111111
Precision (weighted): 0.3273168326840787
Recall (weighted): 0.2911111111111111
F1-score (weighted): 0.23920953272935877

Отчет по классам:
                                             precision    recall  f1-score   support

                   гранты субсидии конкурсы       0.60      0.70      0.65       100
 пожертвования от физических лиц (напрямую)       0.14      0.34      0.20       100
пожертвования от юридических лиц (напрямую)       0.21      0.63      0.31       100
              пожертвования через платформы       0.00      0.00      0.00       100
                            продажа товаров       0.31      0.16      0.21       100
                              продажа услуг       0.00      0.00      0.00       100
                 прочие недоходные операции       0.19      0.10      0.13       100
                          финансовые доходы       1.00      0.05      0.10       100
           членские и учредительские взносы       0.49      0.64  

/opt/anaconda3/envs/datasphere_cli_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda3/envs/datasphere_cli_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda3/envs/datasphere_cli_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(re

In [8]:
y_true.value_counts()

universal_category                         
продажа услуг                                  105
пожертвования от юридических лиц (напрямую)    101
гранты субсидии конкурсы                       100
пожертвования через платформы                  100
прочие недоходные операции                     100
финансовые доходы                              100
членские и учредительские взносы               100
пожертвования от физических лиц (напрямую)      99
продажа товаров                                 95
Name: count, dtype: int64